# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

The paper is *The State of AI-Driven SEO* (FlyRank, March 2026). I read it as a public study that states its own limits (observational design, no p-values or confidence intervals, an exploratory ML appendix). I chose the two findings closest to what I built: a decay label and a model that ranks pages. The questions are the ones I had to ask of my own work, asked in the way I would want my own work reviewed.

### Finding A: "The Anatomy of Growing Content" (Finding #1)

**What it says:** growing pages are longer (about 3.2K vs 2.3K words) and younger (184 vs 230 days) than declining pages, from 74,187 rising and 45,272 falling pages.

**Question: where does the label come from?** The paper defines rising and falling from the change in impressions between the last 30 days and the previous 30 days (up above +10%, down below -10%). I would ask whether a minimum volume is required before a page gets a direction. In my own data, percentage changes on pages with very few impressions were often forced by the data: a page with one impression in 90 days could only be labeled as falling. With a ±10% rule, small pages can swing easily. I would also ask whether "younger" reflects page structure or the early growth phase the paper itself describes for the first 90 days (Finding #2).

**What would settle it:** the same comparison restricted to pages above a minimum impression count, and within the same age band.

### Finding B: "What Predicts Growth?" (ML appendix, 71% holdout accuracy)

**What it says:** a logistic regression separates growing from declining pages with 71% accuracy on a held-out 20% of the sample.

**Question: does the validation design support the claim?** Three things I could not find in the paper:
1. **A base rate.** In the paper's own comparison, 62.1% of rising-or-falling pages are rising (74,187 of 119,459). If the appendix sample has a similar mix, always guessing "growing" would score about 62%, so 71% would be roughly 9 points above that. The appendix does not state its class balance.
2. **How the 80/20 split was made.** The paper has 57 brands, and pages from one brand tend to resemble each other. A random split across pages can let a model recognize brands it has already seen; a split by brand answers the question "does it work on a brand it never saw?". The paper does not say which was used.
3. **Which window the features cover.** The paper says days visible and recent impressions are among the strongest signals, while the label is itself a change in impressions. If those features cover a window that overlaps the label's window, part of the accuracy could be the label being partly visible in the inputs.

**What would settle it:** accuracy reported next to the majority-class rate, a split by brand, and features taken from a window that ends before the label window begins. I test these three things on my own model in sections 2 and 3.

In [1]:
# Class balance implied by the paper's own Finding #1 counts (taken from the paper, not from my data)
rising, falling = 74187, 45272
share_rising = rising / (rising + falling)
print(f"Rising pages: {share_rising:.1%} of rising-or-falling pages ({rising:,} of {rising + falling:,})")
print(f"Reported accuracy 71% minus majority-class rate: {71 - share_rising * 100:.1f} points")

Rising pages: 62.1% of rising-or-falling pages (74,187 of 119,459)
Reported accuracy 71% minus majority-class rate: 8.9 points


## 2. My model under an honest split (before/after)

**Before:** a random 5-fold split over pages. Pages from one client sit on both the training side and the test side, so the model is tested on clients it has already learned from.

**After:** the 5-fold split grouped by `client_id`, as in ML-08. Every client is on one side only, so the test asks the honest question: does it work on a client it has never seen?

Everything else is identical in both runs (same 21,758-page review population, same 34 features, same models and settings, seed 42), so the difference between the two rows is the effect of the split alone. The base rate is printed next to the results, and the last column shows how many test pages belong to a client that also appears in training.

In [2]:
!test -d repo || git clone --depth 1 https://github.com/flyrank-bih/flyrank-ml-internship-starter.git repo
!cp -r repo/data .

Cloning into 'repo'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 94 (delta 12), reused 29 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 1.83 MiB | 4.26 MiB/s, done.
Resolving deltas: 100% (12/12), done.


In [3]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[df["trend_pct"].notna()].reset_index(drop=True)
df["is_decayed"] = ((df["trend_direction"] == "down") & (df["trend_pct"] <= -30)).astype(int)
df = df[df["impressions_90d"] >= 100].reset_index(drop=True)      # review population, as in ML-08

FORBIDDEN = ["trend_direction", "trend_pct", "is_decayed"] + \
            [c for c in df.columns if c.endswith(("_last_30d", "_prev_30d"))]
FEATURES = [c for c in df.columns if c not in FORBIDDEN + ["content_id", "client_id"]]

X = df[FEATURES].copy()
y = df["is_decayed"].values
groups = df["client_id"].values
ids = df["content_id"].values
NUMERIC = X.select_dtypes(include="number").columns.tolist()
CATEGORICAL = [c for c in FEATURES if c not in NUMERIC]
for c in CATEGORICAL:
    X[c] = X[c].astype("category")
X_lr = X.copy()
for c in CATEGORICAL:
    X_lr[c] = X_lr[c].astype(object)

def make_logistic():
    prep = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL)])
    return Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=1000, random_state=SEED))])

def make_boosting():
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                          categorical_features="from_dtype", random_state=SEED)

def p_at_k(y_true, score, k, tie_ids):
    return y_true[np.lexsort((tie_ids, -score))[:k]].mean()

print("Review population:", f"{len(df):,}", "pages |", len(set(groups)), "clients | base rate:", round(y.mean(), 3))
print("Features:", len(FEATURES), "| sklearn", sklearn.__version__, "| seed", SEED)

Review population: 21,758 pages | 30 clients | base rate: 0.514
Features: 34 | sklearn 1.6.1 | seed 42


In [4]:
splits = {
    "Before: random split":     list(KFold(n_splits=5, shuffle=True, random_state=SEED).split(df)),
    "After: grouped by client": list(GroupKFold(n_splits=5).split(df, y, groups)),
}
rows = []
for split_name, folds in splits.items():
    # how many test pages belong to a client that also appears in training?
    seen = np.mean(np.concatenate([np.isin(groups[te], np.unique(groups[tr])) for tr, te in folds]))
    for model_name, make, features in [("Logistic Regression", make_logistic, X_lr),
                                       ("Gradient boosting", make_boosting, X)]:
        oof = np.zeros(len(y))
        for tr, te in folds:
            oof[te] = make().fit(features.iloc[tr], y[tr]).predict_proba(features.iloc[te])[:, 1]
        p500 = p_at_k(y, oof, 500, ids)
        rows.append({"split": split_name, "model": model_name,
                     "P@500": round(p500, 3), "lift@500": round(p500 / y.mean(), 2),
                     "AUC": round(roc_auc_score(y, oof), 3),
                     "test pages from clients seen in training": f"{seen:.0%}"})

result = pd.DataFrame(rows)
print(f"Base rate: {y.mean():.3f} (random ranking gives P@500 of about this)\n")
print(result.to_string(index=False))

Base rate: 0.514 (random ranking gives P@500 of about this)

                   split               model  P@500  lift@500   AUC test pages from clients seen in training
    Before: random split Logistic Regression  0.862      1.68 0.736                                     100%
    Before: random split   Gradient boosting  0.936      1.82 0.775                                     100%
After: grouped by client Logistic Regression  0.796      1.55 0.638                                       0%
After: grouped by client   Gradient boosting  0.710      1.38 0.667                                       0%


**Observed** (21,758 pages, base rate 0.514; the only difference between the two splits is how pages are assigned to folds):

| Model | Split | P@500 | Lift@500 | AUC |
|---|---|---|---|---|
| Logistic Regression | Random | 0.862 | 1.68x | 0.736 |
| Logistic Regression | Grouped by client | 0.796 | 1.55x | 0.638 |
| Gradient boosting | Random | 0.936 | 1.82x | 0.775 |
| Gradient boosting | Grouped by client | 0.710 | 1.38x | 0.667 |

- **The random split is optimistic.** Under it, 100% of test pages belong to a client the model has already seen; under the grouped split, none do. Going from random to grouped lowers P@500 by 0.066 for Logistic Regression and by 0.226 for Gradient boosting, and lowers AUC by about 0.10 for both.
- **The gap is a finding in itself.** It measures how much of the apparent skill comes from recognizing clients instead of pages. Gradient boosting loses much more than Logistic Regression, which is consistent with a more flexible model picking up more client-specific patterns. I did not test that explanation directly.
- **The model ranking reverses.** Under the random split Gradient boosting looks clearly better (0.936 vs 0.862); under the grouped split Logistic Regression is ahead (0.796 vs 0.710). A reader who saw only the random-split numbers would pick the wrong model.
- **What I keep:** only grouped-split numbers are quoted from here on. The random-split rows exist to show the size of the gap, not as results.
- **Link to the paper (Finding B):** the paper's appendix reports a random-style 80/20 split without saying whether pages were grouped by brand. On my data, a random split inflated P@500 by 0.07 to 0.23, so I would ask whether something similar applies there. I cannot tell from the paper, and different data may behave differently.

## 3. Leakage audit

The checklist from the `hunting-leakage-and-validating` skill, applied to my 34 features. All scores use the client-grouped split from section 2, and every score sits next to the base rate.

**Timeline (one snapshot).** The label compares the last 30 days with the prior 30 days. The 90-day totals contain both of those windows, so they are not strictly "before" the label. Page attributes (content type, intent, length, age, keyword demand) are not part of those windows.

**Checks, in the cells below:**
1. **Label-derived columns:** add the two label-window columns (`impressions_last_30d`, `impressions_prev_30d`) on purpose, watch the score jump, then remove them and keep the honest number.
2. **Overlapping windows:** how much of the 90-day total the two label windows make up, and what the models score with and without the 90-day activity numbers.
3. **Product flags:** a check that the source data holds no flag, action or score columns from an existing system.
4. **Population selection:** the review population is chosen with `impressions_90d`, which includes the outcome window. I compare it with a rule that uses only the prior 30 days.

In [5]:
folds = list(GroupKFold(n_splits=5).split(df, y, groups))

def grouped_oof(features, make=make_boosting):
    oof = np.zeros(len(y))
    for tr, te in folds:
        oof[te] = make().fit(features.iloc[tr], y[tr]).predict_proba(features.iloc[te])[:, 1]
    return oof

def summarize(oof):
    return round(roc_auc_score(y, oof), 3), round(p_at_k(y, oof, 500, ids), 3)

LEAK = ["impressions_last_30d", "impressions_prev_30d"]    # the two inputs of trend_pct, i.e. of the label
auc_honest, p_honest = summarize(grouped_oof(X))

X_leak = X.copy()
for c in LEAK:
    X_leak[c] = df[c].values
auc_leak, p_leak = summarize(grouped_oof(X_leak))

X_leak = X_leak.drop(columns=LEAK)                         # remove the leak again
assert list(X_leak.columns) == list(X.columns)
auc_after, p_after = summarize(grouped_oof(X_leak))

print(f"Base rate: {y.mean():.3f}\n")
print(pd.DataFrame([
    {"features": "34 features (honest)",           "AUC": auc_honest, "P@500": p_honest},
    {"features": "+ the two label-window columns",  "AUC": auc_leak,   "P@500": p_leak},
    {"features": "leak removed again",              "AUC": auc_after,  "P@500": p_after},
]).to_string(index=False))

Base rate: 0.514

                      features   AUC  P@500
          34 features (honest) 0.667   0.71
+ the two label-window columns 0.999   1.00
            leak removed again 0.667   0.71


In [6]:
share = (df["impressions_last_30d"] + df["impressions_prev_30d"]) / df["impressions_90d"]
print(f"The two label windows hold a median of {share.median():.0%} of impressions_90d")

ACTIVITY = [c for c in ["impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
                        "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
                        "days_with_impressions", "days_with_sessions", "ctr", "avg_position",
                        "engagement_rate", "scroll_rate", "ai_traffic_pct",
                        "impression_tier", "position_tier"] if c in FEATURES]
ATTRIBUTES = [c for c in FEATURES if c not in ACTIVITY]

def make_logistic_on(cols):
    num = [c for c in cols if c in NUMERIC]
    cat = [c for c in cols if c in CATEGORICAL]
    prep = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat)])
    return Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=1000, random_state=SEED))])

rows = []
for label, cols in [(f"all {len(FEATURES)} features", FEATURES), (f"attributes only ({len(ATTRIBUTES)})", ATTRIBUTES)]:
    for model_name, oof in [("Logistic Regression", grouped_oof(X_lr[cols], lambda c=cols: make_logistic_on(c))),
                            ("Gradient boosting",    grouped_oof(X[cols]))]:
        auc, p = summarize(oof)
        rows.append({"features": label, "model": model_name, "AUC": auc, "P@500": p})
print("\nGrouped split, with and without the 90-day activity numbers:")
print(pd.DataFrame(rows).to_string(index=False))

The two label windows hold a median of 56% of impressions_90d

Grouped split, with and without the 90-day activity numbers:
            features               model   AUC  P@500
     all 34 features Logistic Regression 0.638  0.796
     all 34 features   Gradient boosting 0.667  0.710
attributes only (17) Logistic Regression 0.565  0.580
attributes only (17)   Gradient boosting 0.592  0.484


In [7]:
SOURCE_COLUMNS = pd.read_csv("data/raw/content_refresh_anonymized.csv", nrows=0).columns
print("Flag-like columns in the source data:",
      [c for c in SOURCE_COLUMNS if any(k in c.lower() for k in ["flag", "action", "recommend", "priority", "rule", "reason"])])

labeled = pd.read_csv("data/raw/content_refresh_anonymized.csv")
labeled = labeled[labeled["trend_pct"].notna()].reset_index(drop=True)
labeled["is_decayed"] = ((labeled["trend_direction"] == "down") & (labeled["trend_pct"] <= -30)).astype(int)

def evaluate_population(frame):
    Xp = frame[FEATURES].copy()
    for c in CATEGORICAL:
        Xp[c] = Xp[c].astype("category")
    Xp_lr = Xp.copy()
    for c in CATEGORICAL:
        Xp_lr[c] = Xp_lr[c].astype(object)
    yp, gp, ip = frame["is_decayed"].values, frame["client_id"].values, frame["content_id"].values
    folds_p = list(GroupKFold(n_splits=5).split(frame, yp, gp))
    out = []
    for model_name, make, feats in [("Logistic Regression", make_logistic, Xp_lr), ("Gradient boosting", make_boosting, Xp)]:
        oof = np.zeros(len(yp))
        for tr, te in folds_p:
            oof[te] = make().fit(feats.iloc[tr], yp[tr]).predict_proba(feats.iloc[te])[:, 1]
        top = np.lexsort((ip, -oof))[:500]
        out.append({"model": model_name, "pages": len(yp), "base rate": round(yp.mean(), 3),
                    "P@500": round(yp[top].mean(), 3), "lift@500": round(yp[top].mean() / yp.mean(), 2),
                    "AUC": round(roc_auc_score(yp, oof), 3),
                    "top 500 with zero recent impressions": f"{(frame['impressions_last_30d'].values[top] == 0).mean():.0%}"})
    return out

rows = []
for rule, mask in [("impressions_90d >= 100 (used here)", labeled["impressions_90d"] >= 100),
                   ("impressions_prev_30d >= 100 (past window only)", labeled["impressions_prev_30d"] >= 100)]:
    for r in evaluate_population(labeled[mask].reset_index(drop=True)):
        rows.append({"population rule": rule, **r})
print("\nPopulation rule sensitivity (grouped split):")
print(pd.DataFrame(rows).to_string(index=False))

Flag-like columns in the source data: []

Population rule sensitivity (grouped split):
                               population rule               model  pages  base rate  P@500  lift@500   AUC top 500 with zero recent impressions
            impressions_90d >= 100 (used here) Logistic Regression  21758      0.514  0.796      1.55 0.638                                  12%
            impressions_90d >= 100 (used here)   Gradient boosting  21758      0.514  0.710      1.38 0.667                                   5%
impressions_prev_30d >= 100 (past window only) Logistic Regression  18010      0.519  0.902      1.74 0.626                                   7%
impressions_prev_30d >= 100 (past window only)   Gradient boosting  18010      0.519  0.920      1.77 0.683                                   4%


**Observed** (grouped split, 21,758 pages, base rate 0.514):

**1. Label-derived columns.** With the two label-window columns added, AUC rose from 0.667 to 0.999 and P@500 from 0.710 to 1.00; after removing them both returned to 0.667 and 0.710. The test harness responds to a leak, and the exclusion is necessary.

**2. Overlapping windows.** The two label windows hold a median of 56% of `impressions_90d`, so the 90-day totals are largely the same weeks the label is built from. With page attributes only (17 features, no 90-day activity numbers), AUC falls from 0.638 to 0.565 for Logistic Regression and from 0.667 to 0.592 for Gradient boosting, and P@500 falls from 0.796 to 0.580 and from 0.710 to 0.484 (below the 0.514 base rate). Most of the measured skill therefore sits in activity numbers that overlap the label windows, and page attributes alone carry a weak signal.

**3. Product flags.** The source data has no flag, action, score or reason columns, so no output of an existing system is used as an input.

**4. Population selection.** The review population is chosen with `impressions_90d`, which includes the outcome window. Switching to a rule that uses only the prior 30 days (`impressions_prev_30d >= 100`, 18,010 pages) keeps AUC similar (0.683 vs 0.667 for Gradient boosting, 0.626 vs 0.638 for Logistic Regression) but moves P@500 from 0.710 to 0.920 and from 0.796 to 0.902. The share of the top 500 with zero recent impressions does not explain the difference (7% and 4% under the prior-30-day rule, 12% and 5% under the rule used here). One candidate explanation, not tested here: the 90-day rule keeps pages with fewer than 100 impressions in the prior 30 days, where a -30% change is noisier, so precision is harder to reach.

**Status against the checklist**

| Check | Status |
|---|---|
| Label-derived columns excluded | Pass (tested) |
| No product flags as inputs | Pass |
| Grouped split, out-of-fold, base rate printed | Pass |
| Features strictly before the label window | **Not met**: the 90-day totals overlap the label windows; shown above |
| Population selection free of outcome-window information | **Disclosed**: both rules condition on a window the label uses, and results depend on the rule |
| Sealed or holdout claim | None made |

**What this means for my claims:** the ranking quality (AUC about 0.63 to 0.68) is fairly stable across both population rules, but P@500 is not (0.71 to 0.92). Single precision figures should therefore be read as directional and tied to their population rule.

## 4. Claim rewrite

Before rewriting any claim, two pieces of evidence:

1. **Failure examples:** for the model with the higher P@500, which of its top-500 pages did not decay, what actually happened to them, and which decayed pages it ranked low.
2. **Uncertainty:** a 95% interval for lift@500 from resampling the 30 clients, and paired differences between methods on the same resampled clients. This shows how much of a difference between two methods could come from which clients happen to be in the data.

The claim rewrite follows after the output.

In [8]:
oof = {"Logistic Regression": grouped_oof(X_lr, make_logistic), "Gradient boosting": grouped_oof(X)}
best = max(oof, key=lambda n: p_at_k(y, oof[n], 500, ids))
order = np.lexsort((ids, -oof[best]))
rank_of = np.empty(len(order), dtype=int)
rank_of[order] = np.arange(1, len(order) + 1)

top = order[:500]
false_pos = top[y[top] == 0]
print(f"{best}: {len(false_pos)} of the top 500 did not decay (false positives)")

# What actually happened to them (the 30-day trend columns are used for diagnosis only)
print("\nTheir recorded trend_direction:")
print(df.loc[false_pos, "trend_direction"].value_counts().to_string())
down = df.loc[false_pos][df.loc[false_pos, "trend_direction"] == "down"]
print(f"Of the 'down' ones, median trend_pct = {down['trend_pct'].median():.1f}% (the decay cutoff is -30%)")

cols = ["impressions_90d", "days_with_impressions", "avg_position", "ctr"]
examples = df.loc[false_pos[:5], cols + ["trend_pct"]].copy()
examples.insert(0, "rank", rank_of[false_pos[:5]])
print("\nFive highest-ranked false positives:")
print(examples.round(2).to_string(index=False))

# False negatives: decayed pages ranked in the bottom half
missed = np.where((y == 1) & (rank_of > len(y) // 2))[0]
print(f"\nDecayed pages ranked in the bottom half (false negatives): {len(missed):,} of {int(y.sum()):,} decayed pages")
print(pd.DataFrame({"missed decayed pages": df.loc[missed, cols].median(),
                    "all pages": df[cols].median()}).round(2).to_string())

Logistic Regression: 102 of the top 500 did not decay (false positives)

Their recorded trend_direction:
trend_direction
up        44
stable    40
down      18
Of the 'down' ones, median trend_pct = -25.0% (the decay cutoff is -30%)

Five highest-ranked false positives:
 rank  impressions_90d  days_with_impressions  avg_position  ctr  trend_pct
    1             4238                     75           7.9 0.21       -0.7
    6              123                     38           6.8 0.00       18.2
    9             2377                     36           2.1 0.08    26166.7
   13              184                     16           2.9 0.00      -25.0
   20              216                     24           0.7 0.00      266.7

Decayed pages ranked in the bottom half (false negatives): 4,436 of 11,174 decayed pages
                       missed decayed pages  all pages
impressions_90d                     2530.00    1728.00
days_with_impressions                 88.00      88.00
avg_position      

In [9]:
flagged = ((df["impressions_90d"] >= 100) & (df["impressions_90d"] < 500) & (df["days_since_last_update"] >= 91))
baseline_score = np.where(flagged, df["days_since_last_update"].clip(upper=365) / 365 * (500 - df["impressions_90d"]) / 400, 0.0)
scores = {"Baseline (Week-4 rule)": baseline_score, **oof}

rng = np.random.default_rng(SEED)
client_ids = np.unique(groups)
rows_by_client = {c: np.where(groups == c)[0] for c in client_ids}
N_BOOT = 1000
lift = {name: [] for name in scores}
for _ in range(N_BOOT):
    pick = rng.choice(client_ids, size=len(client_ids), replace=True)
    idx = np.concatenate([rows_by_client[c] for c in pick])
    yb, ib = y[idx], ids[idx]
    for name, s in scores.items():
        lift[name].append(p_at_k(yb, s[idx], 500, ib) / yb.mean())
lift = {name: np.array(v) for name, v in lift.items()}

def interval(a):
    lo, hi = np.percentile(a, [2.5, 97.5])
    return f"{lo:.2f} to {hi:.2f}"

print(f"Lift@500 with a 95% interval from {N_BOOT} resamples of the {len(client_ids)} clients:")
for name, s in scores.items():
    point = p_at_k(y, s, 500, ids) / y.mean()
    print(f"  {name:24} {point:.2f}x   ({interval(lift[name])})")

print("\nPaired differences in lift@500 (same resampled clients):")
for a, b in [("Logistic Regression", "Baseline (Week-4 rule)"),
             ("Gradient boosting", "Baseline (Week-4 rule)"),
             ("Logistic Regression", "Gradient boosting")]:
    d = lift[a] - lift[b]
    print(f"  {a} minus {b}: median {np.median(d):+.2f}, 95% interval {interval(d)}, "
          f"first is higher in {np.mean(d > 0):.0%} of resamples")

Lift@500 with a 95% interval from 1000 resamples of the 30 clients:
  Baseline (Week-4 rule)   1.20x   (0.92 to 1.35)
  Logistic Regression      1.55x   (1.22 to 1.86)
  Gradient boosting        1.38x   (1.04 to 1.74)

Paired differences in lift@500 (same resampled clients):
  Logistic Regression minus Baseline (Week-4 rule): median +0.36, 95% interval 0.00 to 0.75, first is higher in 98% of resamples
  Gradient boosting minus Baseline (Week-4 rule): median +0.21, 95% interval -0.14 to 0.59, first is higher in 85% of resamples
  Logistic Regression minus Gradient boosting: median +0.15, 95% interval 0.01 to 0.29, first is higher in 98% of resamples


**Observed**

**Failure examples (Logistic Regression, the model with the higher P@500).** 102 of its top 500 pages did not decay. Their recorded direction was `up` for 44, `stable` for 40 and `down` for 18; the 18 declines had a median trend_pct of -25.0%, close to the -30% cutoff. So 84 of the 102 (82%) did not fall at all, and only 18 were near misses. The five highest-ranked misses show how this happens: rank 1 has 4,238 impressions and changed by -0.7%; rank 13 declined by 25.0%; ranks 9 and 20 grew by 26,167% and 267%, percentages that come from very small earlier volumes.

**Pages the model missed.** 4,436 of the 11,174 decayed pages (40%) were ranked in the bottom half. They have a higher median volume than all pages (2,530 vs 1,728 impressions) and a weaker average position (15.7 vs 12.2), so the model is least likely to flag decaying pages that are larger and rank further down.

**Uncertainty from resampling the 30 clients (lift@500, 95% interval).** Baseline rule 1.20x (0.92 to 1.35), Logistic Regression 1.55x (1.22 to 1.86), Gradient boosting 1.38x (1.04 to 1.74). In paired comparisons on the same resampled clients:
- Logistic Regression minus the baseline: median +0.36, interval 0.00 to 0.75, higher in 98% of resamples.
- Gradient boosting minus the baseline: median +0.21, interval -0.14 to 0.59, higher in 85% of resamples; I cannot separate this from no difference.
- Logistic Regression minus Gradient boosting: median +0.15, interval 0.01 to 0.29, higher in 98% of resamples.

These intervals reflect only which clients are in the data. They do not include the label cutoff or the population rule, which moved P@500 by more (0.71 to 0.92, section 3).

**Claim rewrite**

My boldest sentence, from the ML-03 notebook title and the ML-08 summary: *"Why ML beats a fixed rule"* and *"both models beat the rule at the top of the list."*

**Rewritten:** On this snapshot, the Logistic Regression queue had a higher lift@500 than the Week-4 rule (1.55x vs 1.20x; higher in 98% of client resamples, median difference +0.36, interval 0.00 to 0.75). For Gradient boosting (1.38x) the difference from the rule is not distinguishable from zero (higher in 85% of resamples, interval -0.14 to 0.59). Both are directional evidence for decision support, not proof that machine learning beats rules.

| Original claim (where) | Problem | Rewritten |
|---|---|---|
| "Logistic Regression matches or beats Gradient boosting" (ML-08) | Hid that the two models rank differently | Logistic Regression had the higher lift@500 (higher in 98% of resamples, median +0.15, interval 0.01 to 0.29); Gradient boosting had the higher pooled AUC (0.667 vs 0.638). |
| "Precision@500 of 0.710 (1.38x lift)" quoted as the result (ML-03, ML-08) | One number, no interval, and it depends on the population rule | Gradient boosting's P@500 was 0.710 (lift 1.38x, interval 1.04 to 1.74) on the 21,758-page review population; a prior-30-day population rule gave 0.920 (section 3), so this describes one definition, not general performance. |
| "The queue ranks pages worth a refresh review" (ML-07) | The rule's own lift is close to chance for some client mixes | In the Week-4 rule's top 500, 61.8% decayed (1.20x, interval 0.92 to 1.35, which includes 1.0). It is a review queue; it is not evidence that the flagged pages need a refresh. |
| "The model predicts content decay" (ML-03, ML-08) | The 90-day numbers overlap the label windows | The model ranks pages by how likely they are to be labeled decaying in a snapshot. Without the 90-day activity numbers, AUC fell to 0.565 to 0.592 (section 3), so most of its measured skill is detecting declines already visible in recent activity, not forecasting future decay. |

**Why I kept intervals:** the paper reports no p-values or confidence intervals and states that choice openly. My own results showed that comparisons I first read as clear (Gradient boosting over the rule) were not, which is why I added them.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.